# Feature Engineering

Builds a unified feature set from the preprocessed weekly data.
Output: `features_final.csv`

In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load

In [14]:
df = pd.read_csv('../Initial Data/sales_preprocessed_weekly.csv')
df['week_start'] = pd.to_datetime(df['week_start'])
df = df.sort_values(['product_id', 'week_start']).reset_index(drop=True)

weather  = pd.read_csv('../Initial Data/weather_weekly.csv')
holidays = pd.read_csv('../Initial Data/holiday_weekly.csv')
covid    = pd.read_csv('../Initial Data/covid_weekly.csv')
school = pd.read_csv('../Initial Data/school_holidays_weekly.csv')

df['year'] = df['week_start'].dt.isocalendar().year.astype(int)
df['week'] = df['week_start'].dt.isocalendar().week.astype(int)

print(f"Sales:    {df.shape}  |  Products: {df['product_id'].nunique()}")
print(f"Weather:  {weather.shape}")
print(f"Holidays: {holidays.shape}")
print(f"Covid:    {covid.shape}")

Sales:    (19771, 5)  |  Products: 1027
Weather:  (209, 9)
Holidays: (209, 11)
Covid:    (209, 4)


## 2. Merge External Sources

In [15]:
df = df.merge(weather,  on=['year', 'week'], how='left')
df = df.merge(holidays, on=['year', 'week'], how='left')
df = df.merge(covid,    on=['year', 'week'], how='left')
df = df.merge(school, on=['year', 'week'], how='left')

print(df.shape)
print("\nNull counts after merge:")
print(df.isnull().sum()[df.isnull().sum() > 0])

(19771, 29)

Null counts after merge:
temp_mean              4595
temp_min               4595
temp_max               4595
precip_sum             4595
sunshine_sum           4595
temp_anomaly           4595
heavy_rain             4595
has_holiday               7
min_days_to_holiday       7
hol_ascension             7
hol_christmas             7
hol_easter                7
hol_kings_day             7
hol_liberation_day        7
hol_new_year              7
hol_pentecost             7
lockdown_days             7
lockdown                  7
school_holiday            7
school_autumn             7
school_christmas          7
school_may                7
school_spring             7
school_summer             7
dtype: int64


## 3. Imputation

In [16]:
# Weather: fill by week-of-year average
weather_cols = ['temp_mean', 'temp_min', 'temp_max',
                'precip_sum', 'sunshine_sum', 'temp_anomaly', 'heavy_rain']
for col in weather_cols:
    week_avg = df.groupby('week')[col].transform('mean')
    df[col] = df[col].fillna(week_avg)

# Holidays and COVID: missing → 0
holiday_cols = ['has_holiday', 'min_days_to_holiday',
                'hol_ascension', 'hol_christmas', 'hol_easter',
                'hol_kings_day', 'hol_liberation_day',
                'hol_new_year', 'hol_pentecost']
df[holiday_cols] = df[holiday_cols].fillna(0)
df[['lockdown', 'lockdown_days']] = df[['lockdown', 'lockdown_days']].fillna(0)

print("Null counts after imputation:")
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.any() else "None")

Null counts after imputation:
school_holiday      7
school_autumn       7
school_christmas    7
school_may          7
school_spring       7
school_summer       7
dtype: int64


## 4. Calendar Features

In [17]:
df['month']          = df['week_start'].dt.month
df['week_of_year']   = df['week_start'].dt.isocalendar().week.astype(int)
df['is_month_start'] = (df['week_start'].dt.day <= 7).astype(int)
df['is_month_end']   = (df['week_start'].dt.day >= 24).astype(int)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['week_sin']  = np.sin(2 * np.pi * df['week_of_year'] / 52)
df['week_cos']  = np.cos(2 * np.pi * df['week_of_year'] / 52)

print(df[['month', 'week_of_year', 'month_sin', 'month_cos', 'week_sin', 'week_cos']].head(5))

   month  week_of_year     month_sin     month_cos      week_sin      week_cos
0      6            27  1.224647e-16 -1.000000e+00 -1.205367e-01 -9.927089e-01
1      9            39 -1.000000e+00 -1.836970e-16 -1.000000e+00 -1.836970e-16
2     12            52 -2.449294e-16  1.000000e+00  6.432491e-16  1.000000e+00
3      6            26  1.224647e-16 -1.000000e+00 -3.216245e-16 -1.000000e+00
4      8            35 -8.660254e-01 -5.000000e-01 -8.854560e-01 -4.647232e-01


## 5. Lag & Rolling Features

In [18]:
for lag in [1, 2, 4, 8]:
    df[f'lag_{lag}'] = df.groupby('product_id')['sku_sold'].transform(
        lambda x: x.shift(lag)
    )

for window in [4, 8]:
    df[f'rolling_mean_{window}'] = df.groupby('product_id')['sku_sold'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    df[f'rolling_std_{window}'] = df.groupby('product_id')['sku_sold'].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).std()
    )

df[['rolling_std_4', 'rolling_std_8']] = df[['rolling_std_4', 'rolling_std_8']].fillna(0)

before = len(df)
df = df.dropna(subset=['lag_1', 'lag_2', 'lag_4', 'lag_8']).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows due to lag NaN")
print(f"Final shape: {df.shape}  |  Products: {df['product_id'].nunique()}")

Dropped 4302 rows due to lag NaN
Final shape: (15469, 45)  |  Products: 309


## 6. Feature List

In [19]:
FEATURE_COLS = [
    # Lag
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    # Rolling statistics
    'rolling_mean_4', 'rolling_mean_8',
    'rolling_std_4',  'rolling_std_8',
    # Calendar
    'month', 'week_of_year',
    'is_month_start', 'is_month_end',
    'month_sin', 'month_cos',
    'week_sin',  'week_cos',
    # Dutch public holidays
    'has_holiday', 'min_days_to_holiday',
    'hol_new_year', 'hol_easter',
    'hol_kings_day', 'hol_liberation_day',
    'hol_ascension', 'hol_pentecost',
    'hol_christmas',
    # KNMI weather
    'temp_mean', 'temp_min', 'temp_max',
    'precip_sum', 'sunshine_sum',
    'temp_anomaly', 'heavy_rain',
    # COVID
    'lockdown', 'lockdown_days',
    # School holiday
    'school_holiday',
    'school_autumn', 'school_christmas', 'school_may',
    'school_spring', 'school_summer',
]

TARGET = 'sku_sold'

print(f"Total features: {len(FEATURE_COLS)}")
print("\nNull check:")
null_counts = df[FEATURE_COLS].isnull().sum()
print(null_counts[null_counts > 0] if null_counts.any() else "None")

Total features: 40

Null check:
None


## 7. Export

In [20]:
ID_COLS = ['product_id', 'week_start', 'year', 'week']

df[ID_COLS + FEATURE_COLS + [TARGET]].to_csv('features_final.csv', index=False)
print(f"Saved: features_final.csv  →  {df.shape}")

Saved: features_final.csv  →  (15469, 45)
